# DICOM QC - Development Notebook

Development notebook that uses source code from disk (not installed package).

**Usage:** Restart kernel to pick up code changes.

In [ ]:
import sys
from pathlib import Path

# Use local dicom_qc code (restart kernel to pick up changes)
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

# Clear any cached imports
for mod in list(sys.modules.keys()):
    if 'dicom_qc' in mod:
        del sys.modules[mod]

%matplotlib widget

# Verify we're using local code
import dicom_qc
print(f"Using dicom_qc from: {dicom_qc.__file__}")

In [ ]:
from dicom_qc import QuickCheck

# ============================================================
# CONFIGURE THIS PATH
# ============================================================

# Directory containing DICOM files (searches recursively for *.dcm)
DATA_DIR = Path('/path/to/your/dicom/data')

# ============================================================

# Initialize (state saved in DATA_DIR/_dicom_qc/)
qc = QuickCheck(DATA_DIR)

In [ ]:
# Load previous state if exists, then discover files
qc.load_if_exists()
qc.discover()

print(f"Found {len(qc.patients)} patients, {len(qc.get_all_series())} series")

In [ ]:
# Process series (run QC checks, generate thumbnails)
qc.process_all()

In [ ]:
# Generate HTML reports - test both modes

# 1. Embedded (single self-contained file)
html_path, zip_path = qc.generate_html_report(
    DATA_DIR / 'qc_report_embedded.html',
    embed_thumbnails=True
)
print(f"Embedded report: {html_path}")
assert zip_path is None, "Embedded report should not have zip"
assert html_path.exists(), "Embedded HTML should exist"

# 2. Multi-page with zip (for large datasets / sharing)
html_path, zip_path = qc.generate_html_report(
    DATA_DIR / 'qc_report_multipage.html',
    embed_thumbnails=False
)
print(f"Multi-page report: {html_path}")
print(f"For sharing: {zip_path}")
assert zip_path is not None, "Multi-page report should have zip"
assert zip_path.exists(), "Zip file should exist"
assert html_path.exists(), "Index HTML should exist"
assert html_path.name == "index.html", "Multi-page should return index.html"

print("\nBoth report modes working correctly!")

In [ ]:
# Interactive review
qc.display()

## Tips

### Re-process
```python
qc.process_all(reprocess=True)  # Re-run all
qc.process_all(retry_errors=True)  # Only retry errors
```

### Start fresh
```python
qc.reset()      # Clears all data and storage
qc.discover()   # Re-scan for DICOM files
```